<a href="https://colab.research.google.com/github/rajanani6767/ML/blob/main/tASK10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

print("Dataset Shape:", X.shape)

Dataset Shape: (569, 30)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
baseline_model = MLPClassifier(max_iter=300, random_state=42)

baseline_model.fit(X_train, y_train)

y_pred_base = baseline_model.predict(X_test)

print("\n===== BASELINE MODEL =====")
print("Accuracy:", accuracy_score(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))


===== BASELINE MODEL =====
Accuracy: 0.9736842105263158
              precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [5]:
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50)],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'batch_size': [16, 32, 64],
    'activation': ['relu', 'tanh']
}

random_search = RandomizedSearchCV(
    MLPClassifier(max_iter=300, random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\n===== RANDOM SEARCH BEST PARAMETERS =====")
print(random_search.best_params_)


===== RANDOM SEARCH BEST PARAMETERS =====
{'learning_rate_init': 0.01, 'hidden_layer_sizes': (100, 50), 'batch_size': 64, 'activation': 'relu'}


In [6]:
grid_search = GridSearchCV(
    MLPClassifier(max_iter=300, random_state=42),
    param_grid=param_dist,
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n===== GRID SEARCH BEST PARAMETERS =====")
print(grid_search.best_params_)


===== GRID SEARCH BEST PARAMETERS =====
{'activation': 'relu', 'batch_size': 16, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.1}


In [7]:
best_model = random_search.best_estimator_   # or grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

print("\n===== TUNED MODEL =====")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))


===== TUNED MODEL =====
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        43
           1       0.97      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114



In [8]:
base_acc = accuracy_score(y_test, y_pred_base)
tuned_acc = accuracy_score(y_test, y_pred_tuned)

print("\n===== COMPARISON =====")
print("Baseline Accuracy:", base_acc)
print("Tuned Accuracy:", tuned_acc)

if tuned_acc > base_acc:
    print("✅ Performance Improved after Tuning")
else:
    print("⚠️ No Improvement")


===== COMPARISON =====
Baseline Accuracy: 0.9736842105263158
Tuned Accuracy: 0.9649122807017544
⚠️ No Improvement
